# Week 2 · Class 4 — Data Collection for RAG

Scrape real web data, parse a PDF, chunk the text, and feed a real chunk into the `docs` node you
built in Class 3.

## Before you start

This notebook is standalone and runs top to bottom on its own. Section 3 quickly rebuilds just the
one piece of your Class 3 agent we need — the context-aware `docs` chain — so the final wiring
step actually runs here without repeating the full LangGraph build.

Get a free API key at [console.groq.com](https://console.groq.com) before Section 2.


## Section 1 — Install packages


In [ ]:
!pip install -q langchain-core langchain-groq requests beautifulsoup4 pymupdf


## Section 2 — Groq API key


In [ ]:
import os
from getpass import getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

print("GROQ_API_KEY set:", bool(os.environ.get("GROQ_API_KEY")))


## Section 3 — Recap: the `docs` chain from Class 3

This is the same prompt and logic as the `docs_node` you built in Class 3 — condensed to a single
function so this notebook can run on its own. If you already have your Class 3 agent open in
another tab, you can skip straight to Section 4 and come back to wire data into your real
`agent` there instead.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

MODEL = "llama-3.3-70b-versatile"
llm = ChatGroq(model=MODEL, temperature=0.3)

docs_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using ONLY the reference text below. "
               "If the answer isn't in the text, say you don't have that information."),
    ("human", "Reference text:\n{context}\n\nQuestion: {question}"),
])

def answer_from_context(question: str, context: str) -> str:
    result = (docs_prompt | llm).invoke({"question": question, "context": context})
    return result.content

# Quick sanity check with made-up context
print(answer_from_context(
    "What does Groq build?",
    "Groq is a company that builds fast AI inference hardware called LPUs.",
))


## Section 4 — Web scraping setup

We use [quotes.toscrape.com](https://quotes.toscrape.com) — a site built for scraping practice.


In [ ]:
import requests
from bs4 import BeautifulSoup

SCRAPE_URL = "https://quotes.toscrape.com/"
response = requests.get(SCRAPE_URL, timeout=10)
response.raise_for_status()
print("Status:", response.status_code)
soup = BeautifulSoup(response.text, "html.parser")


## Section 5 — Extract quotes


In [ ]:
quotes = []
for item in soup.select("div.quote"):
    text_el = item.select_one("span.text")
    author_el = item.select_one("small.author")
    tags = [t.get_text(strip=True) for t in item.select("a.tag")]
    if text_el and author_el:
        quotes.append({
            "text": text_el.get_text(strip=True),
            "author": author_el.get_text(strip=True),
            "tags": tags,
        })

print(f"Scraped {len(quotes)} quotes")
for q in quotes[:3]:
    print(f"  - {q['text'][:60]}... — {q['author']}")


## Section 6 — Scrape ethics reminder

- Check `robots.txt` before scraping any site
- Add `time.sleep(1)` between requests on multi-page scrapes
- Use practice sites or APIs in production


In [ ]:
import urllib.robotparser

rp = urllib.robotparser.RobotFileParser()
rp.set_url("https://quotes.toscrape.com/robots.txt")
rp.read()
print("Can fetch /:", rp.can_fetch("*", SCRAPE_URL))


## Section 7 — Save scraped data


In [ ]:
import json
from pathlib import Path

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

quotes_path = data_dir / "scraped_quotes.json"
with open(quotes_path, "w", encoding="utf-8") as f:
    json.dump(quotes, f, indent=2, ensure_ascii=False)

print(f"Saved to {quotes_path}")


## Section 8 — Stretch: books.toscrape.com

Scrape book titles and prices from the first page (extend to pagination in Section 9).


In [ ]:
BOOKS_URL = "https://books.toscrape.com/"
books_resp = requests.get(BOOKS_URL, timeout=10)
books_resp.raise_for_status()
books_soup = BeautifulSoup(books_resp.text, "html.parser")

books = []
for article in books_soup.select("article.product_pod"):
    title = article.select_one("h3 a")
    price = article.select_one("p.price_color")
    if title and price:
        books.append({
            "title": title.get("title", title.get_text(strip=True)),
            "price": price.get_text(strip=True),
        })

print(f"Scraped {len(books)} books")
for b in books[:3]:
    print(f"  - {b['title']} — {b['price']}")


## Section 9 — Stretch: pagination

Follow `next` links to scrape more pages (cap at 3 pages to be polite).


In [ ]:
import time

def scrape_book_page(url: str) -> list[dict]:
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    page_soup = BeautifulSoup(resp.text, "html.parser")
    page_books = []
    for article in page_soup.select("article.product_pod"):
        title = article.select_one("h3 a")
        price = article.select_one("p.price_color")
        if title and price:
            page_books.append({
                "title": title.get("title", title.get_text(strip=True)),
                "price": price.get_text(strip=True),
            })
    next_link = page_soup.select_one("li.next a")
    next_url = None
    if next_link and next_link.get("href"):
        next_url = "https://books.toscrape.com/" + next_link["href"].lstrip("/")
    return page_books, next_url

all_books = []
url = BOOKS_URL
for _ in range(3):
    page_books, url = scrape_book_page(url)
    all_books.extend(page_books)
    if not url:
        break
    time.sleep(1)

books_path = data_dir / "scraped_books.json"
with open(books_path, "w", encoding="utf-8") as f:
    json.dump(all_books, f, indent=2, ensure_ascii=False)
print(f"Saved {len(all_books)} books to {books_path}")


## Section 10 — PDF text extraction

Use the sample PDF in `week-2/data/sample.pdf` or upload your own document.


In [ ]:
import fitz  # PyMuPDF

pdf_path = Path("data/sample.pdf")
if not pdf_path.exists():
    alt = Path("week-2/data/sample.pdf")
    pdf_path = alt if alt.exists() else pdf_path

if pdf_path.exists():
    doc = fitz.open(pdf_path)
    full_text = "".join(page.get_text() for page in doc)
    print(f"Extracted {len(full_text)} characters from {doc.page_count} pages")
    print(full_text[:400])
else:
    print("Upload a PDF to data/sample.pdf or set pdf_path to your file.")
    full_text = ""  # replace after uploading


## Section 11 — Alternative: pdfplumber

Useful for PDFs with tables.


In [ ]:
# !pip install -q pdfplumber
# import pdfplumber
# with pdfplumber.open("data/sample.pdf") as pdf:
#     full_text = "\n".join(page.extract_text() or "" for page in pdf.pages)
# print(full_text[:400])


## Section 12 — Chunking for embeddings


In [ ]:
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> list[str]:
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        piece = text[start:end].strip()
        if piece:
            chunks.append(piece)
        start = end - overlap
    return chunks

chunks = chunk_text(full_text) if full_text else []
print(f"Created {len(chunks)} chunks")
if chunks:
    print("First chunk:", chunks[0][:200], "...")

chunks_path = data_dir / "chunks.json"
with open(chunks_path, "w", encoding="utf-8") as f:
    json.dump(chunks, f, indent=2, ensure_ascii=False)
print(f"Saved to {chunks_path}")


## Section 13 — Wire real data into `docs`

This is the payoff: pass a real scraped/PDF chunk as context, and the answer is grounded in
**your** data instead of the model's memory. If you have your Class 3 `agent` running elsewhere,
replace `answer_from_context(...)` with `agent.invoke({"messages": [...], "context_chunks": ...})`
using that real graph.


In [ ]:
sample_chunk = chunks[0] if chunks else quotes[0]["text"] if quotes else "No data collected yet."

answer = answer_from_context("Summarize this text in one sentence.", sample_chunk)
print("Context used:", sample_chunk[:150], "...")
print("\nAnswer:", answer)


## Section 14 — Deliverable checklist

- [ ] `data/scraped_quotes.json` saved (stretch: `scraped_books.json`)
- [ ] At least one PDF parsed and chunked to `data/chunks.json`
- [ ] `docs` (or your Class 3 `agent`'s `docs` node) answers using a real chunk from your dataset
- [ ] Demo ready (2–3 min): show your Class 3 agent's four intents + one scrape sample + one PDF
      chunk feeding a real, grounded answer

**Week 3:** your chunks become embeddings in FAISS / Qdrant.


In [ ]:
# Final self-check — uncomment and run when done
# assert Path("data/scraped_quotes.json").exists(), "Missing scraped_quotes.json"
# assert Path("data/chunks.json").exists(), "Missing chunks.json"
# print("Deliverable files present. Ready for Week 3!")
